### 1. Generating Synthetic Data

Since no specific dataset was provided, I will create a synthetic dataset with features like `rainfall`, `temperature`, `soil_moisture`, `relative_humidity`, `day_length`, and target variables for `diseases_action`, `weeds_action`, and `crop_planning_action`. These target variables will indicate whether prophylactic action is needed (1) or not (0).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer

# Set a random seed for reproducibility
np.random.seed(42)

# Number of samples
n_samples = 1000

# Generate synthetic features
data = {
    'rainfall': np.random.normal(loc=50, scale=20, size=n_samples), # mm
    'temperature': np.random.normal(loc=25, scale=5, size=n_samples), # Celsius
    'soil_moisture': np.random.normal(loc=40, scale=10, size=n_samples), # %
    'relative_humidity': np.random.normal(loc=70, scale=15, size=n_samples), # %
    'day_length': np.random.normal(loc=12, scale=2, size=n_samples) # hours
}

df = pd.DataFrame(data)

# Ensure non-negative values for physical quantities
df['rainfall'] = df['rainfall'].apply(lambda x: max(0, x))
df['soil_moisture'] = df['soil_moisture'].apply(lambda x: max(0, min(100, x)))
df['relative_humidity'] = df['relative_humidity'].apply(lambda x: max(0, min(100, x)))
df['day_length'] = df['day_length'].apply(lambda x: max(0, x))

# Generate synthetic target variables based on some simple rules
# Diseases: higher humidity, temperature, and rainfall might lead to diseases
df['diseases_action'] = ((df['relative_humidity'] > 80) & (df['temperature'] > 28) & (df['rainfall'] > 60)).astype(int)

# Weeds: sufficient soil moisture and day length might promote weed growth
df['weeds_action'] = ((df['soil_moisture'] > 50) & (df['day_length'] > 10) & (df['rainfall'] > 30)).astype(int)

# Contingent Crop Planning: extreme rainfall or temperature might require planning
df['crop_planning_action'] = (((df['rainfall'] > 80) | (df['rainfall'] < 10)) | ((df['temperature'] > 35) | (df['temperature'] < 15))).astype(int)

print("Synthetic Data Head:")
display(df.head())
print("\nSynthetic Data Info:")
display(df.info())

Synthetic Data Head:


,rainfall,temperature,soil_moisture,relative_humidity,day_length,diseases_action,weeds_action,crop_planning_action
0,59.934283,31.996777,33.248217,41.382887,10.273013,0,0,0
1,47.234714,29.623168,38.554813,57.094225,11.937593,0,0,0
2,62.953771,25.298152,32.075801,63.795917,12.036034,0,0,0
3,80.460597,21.765316,36.920385,98.315315,12.945261,0,0,1
4,45.316933,28.491117,21.063853,78.348297,9.266283,0,0,0



Synthetic Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   rainfall              1000 non-null   float64
 1   temperature           1000 non-null   float64
 2   soil_moisture         1000 non-null   float64
 3   relative_humidity     1000 non-null   float64
 4   day_length            1000 non-null   float64
 5   diseases_action       1000 non-null   int64  
 6   weeds_action          1000 non-null   int64  
 7   crop_planning_action  1000 non-null   int64  
dtypes: float64(5), int64(3)
memory usage: 62.6 KB


None

### 2. Managing Missing Values

I will intentionally introduce some missing values into the dataset and then use `SimpleImputer` to fill them. For numerical features, a common strategy is to impute with the mean or median. Here, I'll use the mean.

In [ ]:
# Introduce some missing values randomly
for col in ['rainfall', 'temperature', 'soil_moisture', 'relative_humidity', 'day_length']:
    missing_indices = np.random.choice(df.index, size=int(0.05 * n_samples), replace=False)
    df.loc[missing_indices, col] = np.nan

print("Data with introduced missing values (first 5 rows with NaNs):")
display(df[df.isnull().any(axis=1)].head())

# Impute missing values using the mean strategy
imputer = SimpleImputer(strategy='mean')

# Apply imputer to the feature columns
feature_cols = ['rainfall', 'temperature', 'soil_moisture', 'relative_humidity', 'day_length']
df[feature_cols] = imputer.fit_transform(df[feature_cols])

print("\nData after imputing missing values (check for NaNs):")
print(df.isnull().sum())

print("\nData Head after imputation:")
display(df.head())

Data with introduced missing values (first 5 rows with NaNs):


,rainfall,temperature,soil_moisture,relative_humidity,day_length,diseases_action,weeds_action,crop_planning_action
1,NaN,29.623168,NaN,57.094225,11.937593,0,0,0
3,80.460597,21.765316,36.920385,NaN,12.945261,0,0,1
10,40.731646,31.586970,NaN,68.595457,12.045262,0,0,0
14,15.501643,33.679819,NaN,61.246010,11.841865,0,0,0
16,NaN,21.742910,37.354852,47.209809,11.038332,0,0,0



Data after imputing missing values (check for NaNs):
rainfall                0
temperature             0
soil_moisture           0
relative_humidity       0
day_length              0
diseases_action         0
weeds_action            0
crop_planning_action    0
dtype: int64

Data Head after imputation:


,rainfall,temperature,soil_moisture,relative_humidity,day_length,diseases_action,weeds_action,crop_planning_action
0,59.934283,31.996777,33.248217,41.382887,10.273013,0,0,0
1,50.604064,29.623168,40.081642,57.094225,11.937593,0,0,0
2,62.953771,25.298152,32.075801,63.795917,12.036034,0,0,0
3,80.460597,21.765316,36.920385,69.562670,12.945261,0,0,1
4,45.316933,28.491117,21.063853,78.348297,9.266283,0,0,0


### 3. Preparing Data for Decision Tree Classifiers

We will separate the features (X) from the target variables (y) for each of our classification tasks.

In [ ]:
# Features (X)
X = df[feature_cols]

# Target variables (y)
y_diseases = df['diseases_action']
y_weeds = df['weeds_action']
y_crop_planning = df['crop_planning_action']

print("Features (X) Head:")
display(X.head())
print("\nDiseases Action Target (y_diseases) Head:")
display(y_diseases.head())

Features (X) Head:


,rainfall,temperature,soil_moisture,relative_humidity,day_length
0,59.934283,31.996777,33.248217,41.382887,10.273013
1,50.604064,29.623168,40.081642,57.094225,11.937593
2,62.953771,25.298152,32.075801,63.795917,12.036034
3,80.460597,21.765316,36.920385,69.562670,12.945261
4,45.316933,28.491117,21.063853,78.348297,9.266283



Diseases Action Target (y_diseases) Head:


,diseases_action
0,0
1,0
2,0
3,0
4,0


### 4. Building and Training Decision Tree Models

We will train three separate Decision Tree Classifiers, one for each prophylactic action.

In [ ]:
# Split data into training and testing sets for each task
X_train, X_test, y_diseases_train, y_diseases_test = train_test_split(X, y_diseases, test_size=0.3, random_state=42)
X_train, X_test, y_weeds_train, y_weeds_test = train_test_split(X, y_weeds, test_size=0.3, random_state=42)
X_train, X_test, y_crop_planning_train, y_crop_planning_test = train_test_split(X, y_crop_planning, test_size=0.3, random_state=42)

# Initialize Decision Tree Classifiers
dt_diseases = DecisionTreeClassifier(random_state=42)
dt_weeds = DecisionTreeClassifier(random_state=42)
dt_crop_planning = DecisionTreeClassifier(random_state=42)

# Train the models
dt_diseases.fit(X_train, y_diseases_train)
dt_weeds.fit(X_train, y_weeds_train)
dt_crop_planning.fit(X_train, y_crop_planning_train)

print("Decision Tree models trained successfully for Diseases, Weeds, and Contingent Crop Planning.")

Decision Tree models trained successfully for Diseases, Weeds, and Contingent Crop Planning.


### 5. Evaluating Models and Making Predictions

Finally, we will evaluate the trained models using accuracy score and demonstrate how to make predictions with new data.

In [ ]:
# Make predictions on the test set
y_diseases_pred = dt_diseases.predict(X_test)
y_weeds_pred = dt_weeds.predict(X_test)
y_crop_planning_pred = dt_crop_planning.predict(X_test)

# Evaluate the models
accuracy_diseases = accuracy_score(y_diseases_test, y_diseases_pred)
accuracy_weeds = accuracy_score(y_weeds_test, y_weeds_pred)
accuracy_crop_planning = accuracy_score(y_crop_planning_test, y_crop_planning_pred)

print(f"Accuracy for Diseases Action: {accuracy_diseases:.2f}")
print(f"Accuracy for Weeds Action: {accuracy_weeds:.2f}")
print(f"Accuracy for Contingent Crop Planning Action: {accuracy_crop_planning:.2f}")

# Example of making a prediction for new data
# Let's say we have new environmental data:
new_data = pd.DataFrame({
    'rainfall': [70, 15, 55], # mm
    'temperature': [30, 18, 26], # Celsius
    'soil_moisture': [60, 30, 45], # %
    'relative_humidity': [85, 60, 72], # %
    'day_length': [13, 9, 11] # hours
})

# Impute any missing values in new data (if applicable, though not for this example)
# new_data_imputed = imputer.transform(new_data) # Use the same imputer fitted on training data

prediction_diseases = dt_diseases.predict(new_data)
prediction_weeds = dt_weeds.predict(new_data)
prediction_crop_planning = dt_crop_planning.predict(new_data)

print("\nPredictions for new data (0 = No Action, 1 = Prophylactic Action):")
print(f"Diseases Action: {prediction_diseases}")
print(f"Weeds Action: {prediction_weeds}")
print(f"Contingent Crop Planning Action: {prediction_crop_planning}")

# Explanation of predictions
def get_action_message(prediction):
    return "needed" if prediction == 1 else "not needed"

for i in range(len(new_data)):
    print(f"\nFor New Data Sample {i+1}:")
    print(f"  - Prophylactic action for Diseases is {get_action_message(prediction_diseases[i])}.")
    print(f"  - Prophylactic action for Weeds is {get_action_message(prediction_weeds[i])}.")
    print(f"  - Prophylactic action for Contingent Crop Planning is {get_action_message(prediction_crop_planning[i])}.")

Accuracy for Diseases Action: 0.98
Accuracy for Weeds Action: 0.98
Accuracy for Contingent Crop Planning Action: 0.98

Predictions for new data (0 = No Action, 1 = Prophylactic Action):
Diseases Action: [0 0 0]
Weeds Action: [1 0 0]
Contingent Crop Planning Action: [0 0 0]

For New Data Sample 1:
  - Prophylactic action for Diseases is not needed.
  - Prophylactic action for Weeds is needed.
  - Prophylactic action for Contingent Crop Planning is not needed.

For New Data Sample 2:
  - Prophylactic action for Diseases is not needed.
  - Prophylactic action for Weeds is not needed.
  - Prophylactic action for Contingent Crop Planning is not needed.

For New Data Sample 3:
  - Prophylactic action for Diseases is not needed.
  - Prophylactic action for Weeds is not needed.
  - Prophylactic action for Contingent Crop Planning is not needed.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer

# Set a random seed for reproducibility
np.random.seed(42)

# Number of samples
n_samples = 1000

# Generate synthetic features
data = {
    'rainfall': np.random.normal(loc=50, scale=20, size=n_samples), # mm
    'temperature': np.random.normal(loc=25, scale=5, size=n_samples), # Celsius
    'soil_moisture': np.random.normal(loc=40, scale=10, size=n_samples), # %
    'relative_humidity': np.random.normal(loc=70, scale=15, size=n_samples), # %
    'day_length': np.random.normal(loc=12, scale=2, size=n_samples) # hours
}

df = pd.DataFrame(data)

# Ensure non-negative values for physical quantities
df['rainfall'] = df['rainfall'].apply(lambda x: max(0, x))
df['soil_moisture'] = df['soil_moisture'].apply(lambda x: max(0, min(100, x)))
df['relative_humidity'] = df['relative_humidity'].apply(lambda x: max(0, min(100, x)))
df['day_length'] = df['day_length'].apply(lambda x: max(0, x))

# Generate synthetic target variables based on some simple rules
# Diseases: higher humidity, temperature, and rainfall might lead to diseases
df['diseases_action'] = ((df['relative_humidity'] > 80) & (df['temperature'] > 28) & (df['rainfall'] > 60)).astype(int)

# Weeds: sufficient soil moisture and day length might promote weed growth
df['weeds_action'] = ((df['soil_moisture'] > 50) & (df['day_length'] > 10) & (df['rainfall'] > 30)).astype(int)

# Contingent Crop Planning: extreme rainfall or temperature might require planning
df['crop_planning_action'] = (((df['rainfall'] > 80) | (df['rainfall'] < 10)) | ((df['temperature'] > 35) | (df['temperature'] < 15))).astype(int)

print("Synthetic Data Head:")
display(df.head())
print("\nSynthetic Data Info:")
display(df.info())

# Introduce some missing values randomly
for col in ['rainfall', 'temperature', 'soil_moisture', 'relative_humidity', 'day_length']:
    missing_indices = np.random.choice(df.index, size=int(0.05 * n_samples), replace=False)
    df.loc[missing_indices, col] = np.nan

print("\nData with introduced missing values (first 5 rows with NaNs):")
display(df[df.isnull().any(axis=1)].head())

# Impute missing values using the mean strategy
imputer = SimpleImputer(strategy='mean')

# Apply imputer to the feature columns
feature_cols = ['rainfall', 'temperature', 'soil_moisture', 'relative_humidity', 'day_length']
df[feature_cols] = imputer.fit_transform(df[feature_cols])

print("\nData after imputing missing values (check for NaNs):")
print(df.isnull().sum())

print("\nData Head after imputation:")
display(df.head())

# Features (X)
X = df[feature_cols]

# Target variables (y)
y_diseases = df['diseases_action']
y_weeds = df['weeds_action']
y_crop_planning = df['crop_planning_action']

print("\nFeatures (X) Head:")
display(X.head())
print("\nDiseases Action Target (y_diseases) Head:")
display(y_diseases.head())

# Split data into training and testing sets for each task
X_train, X_test, y_diseases_train, y_diseases_test = train_test_split(X, y_diseases, test_size=0.3, random_state=42)
X_train, X_test, y_weeds_train, y_weeds_test = train_test_split(X, y_weeds, test_size=0.3, random_state=42)
X_train, X_test, y_crop_planning_train, y_crop_planning_test = train_test_split(X, y_crop_planning, test_size=0.3, random_state=42)

# Initialize Decision Tree Classifiers
dt_diseases = DecisionTreeClassifier(random_state=42)
dt_weeds = DecisionTreeClassifier(random_state=42)
dt_crop_planning = DecisionTreeClassifier(random_state=42)

# Train the models
dt_diseases.fit(X_train, y_diseases_train)
dt_weeds.fit(X_train, y_weeds_train)
dt_crop_planning.fit(X_train, y_crop_planning_train)

print("\nDecision Tree models trained successfully for Diseases, Weeds, and Contingent Crop Planning.")

# Make predictions on the test set
y_diseases_pred = dt_diseases.predict(X_test)
y_weeds_pred = dt_weeds.predict(X_test)
y_crop_planning_pred = dt_crop_planning.predict(X_test)

# Evaluate the models
accuracy_diseases = accuracy_score(y_diseases_test, y_diseases_pred)
accuracy_weeds = accuracy_score(y_weeds_test, y_weeds_pred)
accuracy_crop_planning = accuracy_score(y_crop_planning_test, y_crop_planning_pred)

print(f"\nAccuracy for Diseases Action: {accuracy_diseases:.2f}")
print(f"Accuracy for Weeds Action: {accuracy_weeds:.2f}")
print(f"Accuracy for Contingent Crop Planning Action: {accuracy_crop_planning:.2f}")

# Example of making a prediction for new data
# Let's say we have new environmental data:
new_data = pd.DataFrame({
    'rainfall': [70, 15, 55], # mm
    'temperature': [30, 18, 26], # Celsius
    'soil_moisture': [60, 30, 45], # %
    'relative_humidity': [85, 60, 72], # %
    'day_length': [13, 9, 11] # hours
})

# Impute any missing values in new data (if applicable, though not for this example)
# new_data_imputed = imputer.transform(new_data) # Use the same imputer fitted on training data

prediction_diseases = dt_diseases.predict(new_data)
prediction_weeds = dt_weeds.predict(new_data)
prediction_crop_planning = dt_crop_planning.predict(new_data)

print("\nPredictions for new data (0 = No Action, 1 = Prophylactic Action):")
print(f"Diseases Action: {prediction_diseases}")
print(f"Weeds Action: {prediction_weeds}")
print(f"Contingent Crop Planning Action: {prediction_crop_planning}")

# Explanation of predictions
def get_action_message(prediction):
    return "needed" if prediction == 1 else "not needed"

for i in range(len(new_data)):
    print(f"\nFor New Data Sample {i+1}:")
    print(f"  - Prophylactic action for Diseases is {get_action_message(prediction_diseases[i])}.")
    print(f"  - Prophylactic action for Weeds is {get_action_message(prediction_weeds[i])}.")
    print(f"  - Prophylactic action for Contingent Crop Planning is {get_action_message(prediction_crop_planning[i])}.")

Synthetic Data Head:


,rainfall,temperature,soil_moisture,relative_humidity,day_length,diseases_action,weeds_action,crop_planning_action
0,59.934283,31.996777,33.248217,41.382887,10.273013,0,0,0
1,47.234714,29.623168,38.554813,57.094225,11.937593,0,0,0
2,62.953771,25.298152,32.075801,63.795917,12.036034,0,0,0
3,80.460597,21.765316,36.920385,98.315315,12.945261,0,0,1
4,45.316933,28.491117,21.063853,78.348297,9.266283,0,0,0



Synthetic Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   rainfall              1000 non-null   float64
 1   temperature           1000 non-null   float64
 2   soil_moisture         1000 non-null   float64
 3   relative_humidity     1000 non-null   float64
 4   day_length            1000 non-null   float64
 5   diseases_action       1000 non-null   int64  
 6   weeds_action          1000 non-null   int64  
 7   crop_planning_action  1000 non-null   int64  
dtypes: float64(5), int64(3)
memory usage: 62.6 KB


None


Data with introduced missing values (first 5 rows with NaNs):


...


Features (X) Head:


,rainfall,temperature,soil_moisture,relative_humidity,day_length
0,59.934283,31.996777,33.248217,41.382887,10.273013
1,50.604064,29.623168,40.081642,57.094225,11.937593
2,62.953771,25.298152,32.075801,63.795917,12.036034
3,80.460597,21.765316,36.920385,69.562670,12.945261
4,45.316933,28.491117,21.063853,78.348297,9.266283



Diseases Action Target (y_diseases) Head:


,diseases_action
0,0
1,0
2,0
3,0
4,0



Decision Tree models trained successfully for Diseases, Weeds, and Contingent Crop Planning.

Accuracy for Diseases Action: 0.98
Accuracy for Weeds Action: 0.98
Accuracy for Contingent Crop Planning Action: 0.98

Predictions for new data (0 = No Action, 1 = Prophylactic Action):
Diseases Action: [0 0 0]
Weeds Action: [1 0 0]
Contingent Crop Planning Action: [0 0 0]

For New Data Sample 1:
  - Prophylactic action for Diseases is not needed.
  - Prophylactic action for Weeds is needed.
  - Prophylactic action for Contingent Crop Planning is not needed.

For New Data Sample 2:
  - Prophylactic action for Diseases is not needed.
  - Prophylactic action for Weeds is not needed.
  - Prophylactic action for Contingent Crop Planning is not needed.

For New Data Sample 3:
  - Prophylactic action for Diseases is not needed.
  - Prophylactic action for Weeds is not needed.
  - Prophylactic action for Contingent Crop Planning is not needed.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer

# Set a random seed for reproducibility
np.random.seed(42)


try:
    # Attempt to read the Excel file
    df = pd.read_excel('your_excel_file.xlsx')
    print("Data loaded successfully from 'your_excel_file.xlsx'")
    print("Data Head:")
    display(df.head())
    print("\nData Info:")
    display(df.info())
except FileNotFoundError:
    print("Error: 'your_excel_file.xlsx' not found. Generating synthetic data instead.")
    # Number of samples (used for synthetic data if Excel file is not found)
    n_samples = 1000

    # Generate synthetic features
    data = {
        'rainfall': np.random.normal(loc=50, scale=20, size=n_samples), # mm
        'temperature': np.random.normal(loc=25, scale=5, size=n_samples), # Celsius
        'soil_moisture': np.random.normal(loc=40, scale=10, size=n_samples), # %
        'relative_humidity': np.random.normal(loc=70, scale=15, size=n_samples), # %
        'day_length': np.random.normal(loc=12, scale=2, size=n_samples) # hours
    }
    df = pd.DataFrame(data)

    # Ensure non-negative values for physical quantities (vectorized operations)
    df['rainfall'] = np.maximum(0, df['rainfall'])
    df['soil_moisture'] = np.maximum(0, np.minimum(100, df['soil_moisture']))
    df['relative_humidity'] = np.maximum(0, np.minimum(100, df['relative_humidity']))
    df['day_length'] = np.maximum(0, df['day_length'])

    # Generate synthetic target variables based on some simple rules
    df['diseases_action'] = ((df['relative_humidity'] > 80) & (df['temperature'] > 28) & (df['rainfall'] > 60)).astype(int)
    df['weeds_action'] = ((df['soil_moisture'] > 50) & (df['day_length'] > 10) & (df['rainfall'] > 30)).astype(int)
    df['crop_planning_action'] = (((df['rainfall'] > 80) | (df['rainfall'] < 10)) | ((df['temperature'] > 35) | (df['temperature'] < 15))).astype(int)

    print("Synthetic Data Head:")
    display(df.head())
    print("\nSynthetic Data Info:")
    display(df.info())
# --- MODIFICATION END ---

# Introduce some missing values randomly (only if n_samples is defined, i.e., synthetic data was generated)
# If you loaded your own Excel file and it already has missing values, this step can be skipped or adjusted.
if 'n_samples' in locals():
    for col in ['rainfall', 'temperature', 'soil_moisture', 'relative_humidity', 'day_length']:
        missing_indices = np.random.choice(df.index, size=int(0.05 * n_samples), replace=False)
        df.loc[missing_indices, col] = np.nan

    print("\nData with introduced missing values (first 5 rows with NaNs if any):")
    display(df[df.isnull().any(axis=1)].head())

# Impute missing values using the mean strategy
imputer = SimpleImputer(strategy='mean')

# Apply imputer to the feature columns
feature_cols = ['rainfall', 'temperature', 'soil_moisture', 'relative_humidity', 'day_length']
df[feature_cols] = imputer.fit_transform(df[feature_cols])

print("\nData after imputing missing values (check for NaNs):")
print(df.isnull().sum())

print("\nData Head after imputation:")
display(df.head())

# Features (X)
X = df[feature_cols]

# Target variables (y)
y_diseases = df['diseases_action']
y_weeds = df['weeds_action']
y_crop_planning = df['crop_planning_action']

print("\nFeatures (X) Head:")
display(X.head())
print("\nDiseases Action Target (y_diseases) Head:")
display(y_diseases.head())

# Split data into training and testing sets for each task
X_train, X_test, y_diseases_train, y_diseases_test = train_test_split(X, y_diseases, test_size=0.3, random_state=42)
X_train, X_test, y_weeds_train, y_weeds_test = train_test_split(X, y_weeds, test_size=0.3, random_state=42)
X_train, X_test, y_crop_planning_train, y_crop_planning_test = train_test_split(X, y_crop_planning, test_size=0.3, random_state=42)

# Initialize Decision Tree Classifiers
dt_diseases = DecisionTreeClassifier(random_state=42)
dt_weeds = DecisionTreeClassifier(random_state=42)
dt_crop_planning = DecisionTreeClassifier(random_state=42)

# Train the models
dt_diseases.fit(X_train, y_diseases_train)
dt_weeds.fit(X_train, y_weeds_train)
dt_crop_planning.fit(X_train, y_crop_planning_train)

print("\nDecision Tree models trained successfully for Diseases, Weeds, and Contingent Crop Planning.")

# Make predictions on the test set
y_diseases_pred = dt_diseases.predict(X_test)
y_weeds_pred = dt_weeds.predict(X_test)
y_crop_planning_pred = dt_crop_planning.predict(X_test)

# Evaluate the models
accuracy_diseases = accuracy_score(y_diseases_test, y_diseases_pred)
accuracy_weeds = accuracy_score(y_weeds_test, y_weeds_pred)
accuracy_crop_planning = accuracy_score(y_crop_planning_test, y_crop_planning_pred)

print(f"\nAccuracy for Diseases Action: {accuracy_diseases:.2f}")
print(f"Accuracy for Weeds Action: {accuracy_weeds:.2f}")
print(f"Accuracy for Contingent Crop Planning Action: {accuracy_crop_planning:.2f}")

# Example of making a prediction for new data
# Let's say we have new environmental data:
new_data = pd.DataFrame({
    'rainfall': [70, 15, 55], # mm
    'temperature': [30, 18, 26], # Celsius
    'soil_moisture': [60, 30, 45], # %
    'relative_humidity': [85, 60, 72], # %
    'day_length': [13, 9, 11] # hours
})

# Impute any missing values in new data (if applicable, though not for this example)
# new_data_imputed = imputer.transform(new_data) # Use the same imputer fitted on training data

prediction_diseases = dt_diseases.predict(new_data)
prediction_weeds = dt_weeds.predict(new_data)
prediction_crop_planning = dt_crop_planning.predict(new_data)

print("\nPredictions for new data (0 = No Action, 1 = Prophylactic Action):")
print(f"Diseases Action: {prediction_diseases}")
print(f"Weeds Action: {prediction_weeds}")
print(f"Contingent Crop Planning Action: {prediction_crop_planning}")

# Explanation of predictions
def get_action_message(prediction):
    return "needed" if prediction == 1 else "not needed"

for i in range(len(new_data)):
    print(f"\nFor New Data Sample {i+1}:")
    print(f"  - Prophylactic action for Diseases is {get_action_message(prediction_diseases[i])}.")
    print(f"  - Prophylactic action for Weeds is {get_action_message(prediction_weeds[i])}.")
    print(f"  - Prophylactic action for Contingent Crop Planning is {get_action_message(prediction_crop_planning[i])}.")

Error: 'your_excel_file.xlsx' not found. Generating synthetic data instead.
Synthetic Data Head:


,rainfall,temperature,soil_moisture,relative_humidity,day_length,diseases_action,weeds_action,crop_planning_action
0,59.934283,31.996777,33.248217,41.382887,10.273013,0,0,0
1,47.234714,29.623168,38.554813,57.094225,11.937593,0,0,0
2,62.953771,25.298152,32.075801,63.795917,12.036034,0,0,0
3,80.460597,21.765316,36.920385,98.315315,12.945261,0,0,1
4,45.316933,28.491117,21.063853,78.348297,9.266283,0,0,0



Synthetic Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   rainfall              1000 non-null   float64
 1   temperature           1000 non-null   float64
 2   soil_moisture         1000 non-null   float64
 3   relative_humidity     1000 non-null   float64
 4   day_length            1000 non-null   float64
 5   diseases_action       1000 non-null   int64  
 6   weeds_action          1000 non-null   int64  
 7   crop_planning_action  1000 non-null   int64  
dtypes: float64(5), int64(3)
memory usage: 62.6 KB


None


Data with introduced missing values (first 5 rows with NaNs if any):


...


Features (X) Head:


,rainfall,temperature,soil_moisture,relative_humidity,day_length
0,59.934283,31.996777,33.248217,41.382887,10.273013
1,50.604064,29.623168,40.081642,57.094225,11.937593
2,62.953771,25.298152,32.075801,63.795917,12.036034
3,80.460597,21.765316,36.920385,69.562670,12.945261
4,45.316933,28.491117,21.063853,78.348297,9.266283



Diseases Action Target (y_diseases) Head:


,diseases_action
0,0
1,0
2,0
3,0
4,0



Decision Tree models trained successfully for Diseases, Weeds, and Contingent Crop Planning.

Accuracy for Diseases Action: 0.98
Accuracy for Weeds Action: 0.98
Accuracy for Contingent Crop Planning Action: 0.98

Predictions for new data (0 = No Action, 1 = Prophylactic Action):
Diseases Action: [0 0 0]
Weeds Action: [1 0 0]
Contingent Crop Planning Action: [0 0 0]

For New Data Sample 1:
  - Prophylactic action for Diseases is not needed.
  - Prophylactic action for Weeds is needed.
  - Prophylactic action for Contingent Crop Planning is not needed.

For New Data Sample 2:
  - Prophylactic action for Diseases is not needed.
  - Prophylactic action for Weeds is not needed.
  - Prophylactic action for Contingent Crop Planning is not needed.

For New Data Sample 3:
  - Prophylactic action for Diseases is not needed.
  - Prophylactic action for Weeds is not needed.
  - Prophylactic action for Contingent Crop Planning is not needed.
